<a href="https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Kashaf537/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 206 (delta 83), reused 124 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 2.62 MiB | 6.00 MiB/s, done.
Resolving deltas: 100% (83/83), done.


In [2]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    paper	     scripts   submission
CLAUDE.md  docs		notebooks  README.md	     SETUP.md  work
data	   GUIDE.md	outputs    requirements.txt  skills    workflows


In [4]:
from pathlib import Path

for folder in [
    "data",
    "data/raw",
    "data/processed",
    "scripts",
    "work",
    "work/notebooks",
    "work/outputs"
]:
    p = Path(folder)
    print(f"{folder:35} {'✓ EXISTS' if p.exists() else '✗ MISSING'}")

data                                ✓ EXISTS
data/raw                            ✓ EXISTS
data/processed                      ✗ MISSING
scripts                             ✓ EXISTS
work                                ✓ EXISTS
work/notebooks                      ✓ EXISTS
work/outputs                        ✗ MISSING


In [17]:
from pathlib import Path

dataset = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", dataset.exists())

if dataset.exists():
    import pandas as pd
    df = pd.read_csv(dataset)
    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("first 5 rows")
    print(df.head())

Dataset exists: True
Shape: (30000, 44)
Columns: 44
first 5 rows
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_posit

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use a Random Forest classifier for the Refresh / Content Opportunity Scoring lane.

The Week-4 baseline ranks pages using a simple rule:

`0.5 × staleness score + 0.5 × search-demand score`

where staleness is based on `days_since_last_update` and demand is based on `search_volume`.

For Week 5, Random Forest is used as a more flexible model that can combine multiple content and search signals and capture nonlinear relationships. This fits the lane because the goal is to prioritize pages that are more likely to need attention, rather than simply predict every page equally well.

The target is `is_declining_label`, where a value of 1 means that the observed `trend_direction` is `down`.

I exclude `trend_direction` and `trend_pct` from the model because they contain the outcome information used to construct the target. Identifiers and Week-4 baseline outputs are also excluded to prevent leakage.

The model will be judged against the Week-4 rule-based baseline using the same held-out test population and the same ranking metric, Precision@50. The goal is not to reward complexity; Random Forest is useful only if it improves the ranking of actual declining pages compared with the simpler baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [18]:
from sklearn.model_selection import GroupShuffleSplit

# Create the evaluation target
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


### Why this split is honest

I use a grouped train/test split by `client_id`. This prevents records from the same client appearing in both training and testing.

This is important because multiple content records can belong to the same client. A random row split could therefore make the test set easier by exposing the model to the same client's patterns during training.

The test clients are held out until evaluation. Both the Week-4 baseline and the Random Forest will be evaluated on this same test population.

In [19]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

The Week-4 baseline ranks pages using equal-weight staleness and search demand:

`0.5 × staleness score + 0.5 × demand score`.

For Week 5, I train a Random Forest classifier using content, demand, competition, age, and longer-window engagement signals. Outcome-related fields and identifiers are excluded.

Both approaches are evaluated on the same client-held-out test set.

Because the lane is a prioritization problem, I use Precision@50: the proportion of the top 50 ranked pages that are actually labeled as declining.

The comparison is intended to answer whether the more complex model produces a more useful top-of-queue ranking than the simple Week-4 rule.

In [20]:
# Features available before the decline label is evaluated.
# Exclude identifiers, outcome fields, baseline outputs,
# and short-window trend components that could create leakage.

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_cols = numeric_features + categorical_features

print("Number of features:", len(feature_cols))
print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

missing_features = [
    c for c in feature_cols
    if c not in df.columns
]

print("\nMissing features:", missing_features)

Number of features: 24

Numeric features: 18
Categorical features: 6

Missing features: []


In [21]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining_label"].copy()
y_test = test_df["is_declining_label"].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Declining rate - train:", round(y_train.mean() * 100, 2), "%")
print("Declining rate - test:", round(y_test.mean() * 100, 2), "%")

X_train: (23837, 24)
X_test: (6163, 24)
Declining rate - train: 55.01 %
Declining rate - test: 51.1 %


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

model.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [23]:
rf_scores = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(rf_scores))
print("Minimum score:", round(rf_scores.min(), 4))
print("Maximum score:", round(rf_scores.max(), 4))

Number of test predictions: 6163
Minimum score: 0.03
Maximum score: 0.9667


In [24]:
def precision_at_k(y_true, scores, k=50):
    y_true = pd.Series(y_true).reset_index(drop=True)
    scores = pd.Series(scores).reset_index(drop=True)

    top_k_idx = scores.sort_values(
        ascending=False
    ).head(k).index

    return y_true.loc[top_k_idx].mean()

In [25]:
rf_precision_50 = precision_at_k(
    y_test,
    rf_scores,
    k=50
)

print(
    f"Random Forest Precision@50: "
    f"{rf_precision_50:.3f}"
)

Random Forest Precision@50: 0.520


In [26]:
def percentile_score(train_values, test_values):
    """
    Convert test values into percentile scores using
    the training distribution only.
    """
    train_values = pd.Series(train_values).dropna().sort_values().to_numpy()

    test_values = pd.Series(test_values)

    if len(train_values) == 0:
        return pd.Series(0.0, index=test_values.index)

    import numpy as np

    scores = np.searchsorted(
        train_values,
        test_values.fillna(train_values[0]),
        side="right"
    ) / len(train_values)

    return pd.Series(
        scores,
        index=test_values.index
    ).clip(0, 1)

In [27]:
baseline_test = test_df.copy()

baseline_test["staleness_score"] = percentile_score(
    train_df["days_since_last_update"],
    baseline_test["days_since_last_update"]
)

baseline_test["demand_score"] = percentile_score(
    train_df["search_volume"].fillna(0),
    baseline_test["search_volume"].fillna(0)
)

baseline_test["baseline_score"] = (
    0.5 * baseline_test["staleness_score"]
    + 0.5 * baseline_test["demand_score"]
)

baseline_precision_50 = precision_at_k(
    baseline_test["is_declining_label"],
    baseline_test["baseline_score"],
    k=50
)

print(
    f"Week-4 baseline Precision@50: "
    f"{baseline_precision_50:.3f}"
)

Week-4 baseline Precision@50: 0.360


In [28]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision_50,
        rf_precision_50
    ]
})

comparison["Precision@50"] = comparison["Precision@50"].round(3)

comparison

,Method,Precision@50
0,Week-4 baseline,0.36
1,Random Forest,0.52


In [29]:
improvement = rf_precision_50 - baseline_precision_50

print(
    f"Precision@50 improvement: "
    f"{improvement:+.3f}"
)

Precision@50 improvement: +0.160


In [30]:
if improvement > 0:
    print("Random Forest improves on the Week-4 baseline.")
elif improvement < 0:
    print("Random Forest does not improve on the Week-4 baseline.")
else:
    print("Random Forest matches the Week-4 baseline.")

Random Forest improves on the Week-4 baseline.


In [31]:
rf_results = test_df[
    [
        "content_id",
        "client_id",
        "is_declining_label"
    ]
].copy()

rf_results["rf_score"] = rf_scores

rf_results = rf_results.sort_values(
    "rf_score",
    ascending=False
).reset_index(drop=True)

rf_results["rf_rank"] = rf_results.index + 1

rf_top50 = rf_results.head(50).copy()

print("Random Forest top 50:")
display(rf_top50.head(20))

Random Forest top 50:


,content_id,client_id,is_declining_label,rf_score,rf_rank
0,content_0478bd1b2fb4,client_8527a891e2,0,0.966667,1
1,content_7e1f78c66a44,client_8527a891e2,0,0.966667,2
2,content_e988c1699454,client_8527a891e2,1,0.966667,3
3,content_9ac61c04930e,client_8527a891e2,1,0.963333,4
4,content_f49660e074e9,client_8527a891e2,0,0.963333,5
5,content_2ba626fea4d6,client_8527a891e2,0,0.963333,6
6,content_1d0963b56227,client_4e07408562,0,0.956667,7
7,content_f73c382ac270,client_8527a891e2,1,0.956667,8
8,content_41baf0722ad9,client_8527a891e2,0,0.953333,9
9,content_ee6b25664a03,client_8527a891e2,0,0.953333,10


In [32]:
print(
    "Declining pages in Random Forest top 50:",
    int(rf_top50["is_declining_label"].sum())
)

print(
    "Random Forest Precision@50:",
    round(rf_top50["is_declining_label"].mean(), 3)
)

Declining pages in Random Forest top 50: 26
Random Forest Precision@50: 0.52


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest identified 26 declining pages among its top 50 predictions, giving a Precision@50 of 0.520.

The Week-4 baseline achieved a Precision@50 of 0.360 on the same test set. The Random Forest therefore improved Precision@50 by 0.160, or 16 percentage points.

This suggests that the additional content, demand, age, and engagement features provide useful ranking information beyond the two signals used by the baseline.

However, Precision@50 is not evidence that the model will cause traffic or ranking improvements. It only measures how often the model's highest-ranked pages were labeled as declining in this evaluation dataset.

The remaining 24 pages in the model's top 50 were not labeled as declining. These are false positives from the perspective of this target and are useful for understanding where the model's ranking can be wrong.

In [34]:
# Add actual labels and predictions to the ranked results
import numpy as np

rf_results["predicted_declining"] = (
    rf_results["rf_score"] >= 0.5
).astype(int)

rf_results["error_type"] = np.where(
    (rf_results["predicted_declining"] == 1) &
    (rf_results["is_declining_label"] == 0),
    "False Positive",
    np.where(
        (rf_results["predicted_declining"] == 0) &
        (rf_results["is_declining_label"] == 1),
        "False Negative",
        "Correct"
    )
)

error_summary = (
    rf_results["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

error_summary

,error_type,count
0,Correct,3435
1,False Positive,1632
2,False Negative,1096


In [35]:
top50_false_positives = rf_results[
    (rf_results["rf_rank"] <= 50) &
    (rf_results["is_declining_label"] == 0)
].copy()

print(
    "False positives in top 50:",
    len(top50_false_positives)
)

top50_false_positives[
    [
        "rf_rank",
        "content_id",
        "rf_score",
        "is_declining_label"
    ]
].head(20)

False positives in top 50: 24


,rf_rank,content_id,rf_score,is_declining_label
0,1,content_0478bd1b2fb4,0.966667,0
1,2,content_7e1f78c66a44,0.966667,0
4,5,content_f49660e074e9,0.963333,0
5,6,content_2ba626fea4d6,0.963333,0
6,7,content_1d0963b56227,0.956667,0
8,9,content_41baf0722ad9,0.953333,0
9,10,content_ee6b25664a03,0.953333,0
11,12,content_4d9ab09aad97,0.950000,0
14,15,content_4d9f36001f06,0.940000,0
15,16,content_5585a0e7089c,0.940000,0


In [36]:
false_negatives = rf_results[
    (rf_results["predicted_declining"] == 0) &
    (rf_results["is_declining_label"] == 1)
].copy()

print(
    "False negatives:",
    len(false_negatives)
)

false_negatives[
    [
        "rf_rank",
        "content_id",
        "rf_score",
        "is_declining_label"
    ]
].head(20)

False negatives: 1096


,rf_rank,content_id,rf_score,is_declining_label
3687,3688,content_80c239f9ecc8,0.496667,1
3692,3693,content_4a8ed6268c17,0.496667,1
3694,3695,content_1f1c6b273a44,0.496667,1
3697,3698,content_b7fef277c982,0.496667,1
3698,3699,content_32ec692a6061,0.496667,1
3700,3701,content_bcf1b6a7a986,0.496667,1
3701,3702,content_b80b0230d0a9,0.496667,1
3704,3705,content_69e05c36b3c7,0.496667,1
3708,3709,content_f41264b82587,0.496667,1
3709,3710,content_33ff51dd343d,0.496667,1


In [37]:
error_analysis = test_df[
    [
        "content_id",
        "client_id",
        "search_volume",
        "competition",
        "cpc",
        "content_type",
        "main_intent",
        "word_count",
        "impressions_90d",
        "clicks_90d",
        "engaged_sessions_90d",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "is_declining_label"
    ]
].copy()

error_analysis = error_analysis.merge(
    rf_results[
        [
            "content_id",
            "rf_score",
            "rf_rank",
            "error_type"
        ]
    ],
    on="content_id",
    how="left"
)

error_analysis.sort_values(
    "rf_rank"
).head(20)

,content_id,client_id,search_volume,competition,cpc,content_type,main_intent,word_count,impressions_90d,clicks_90d,...,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,is_declining_label,rf_score,rf_rank,error_type
1511,content_0478bd1b2fb4,client_8527a891e2,10.0,0.00,0.00,keyword article,informational,1625.0,274,1,...,238,103,0.36,27.4,0.00,33.33,0,0.966667,1,False Positive
2791,content_7e1f78c66a44,client_8527a891e2,0.0,0.00,0.00,keyword article,informational,1606.0,1645,1,...,174,92,0.06,47.3,0.00,25.00,0,0.966667,2,False Positive
1316,content_e988c1699454,client_8527a891e2,30.0,0.00,0.00,keyword article,informational,1353.0,2197,0,...,275,104,0.00,21.5,0.00,50.00,1,0.966667,3,Correct
2969,content_9ac61c04930e,client_8527a891e2,10.0,0.00,0.00,keyword article,informational,1504.0,1828,4,...,275,104,0.22,5.6,0.00,10.00,1,0.963333,4,Correct
5876,content_f49660e074e9,client_8527a891e2,0.0,0.00,0.00,keyword article,informational,1577.0,1432,5,...,223,102,0.35,20.2,6.25,5.88,0,0.963333,5,False Positive
4590,content_2ba626fea4d6,client_8527a891e2,10.0,0.00,0.00,keyword article,informational,1405.0,360,0,...,275,104,0.00,7.2,0.00,0.00,0,0.963333,6,False Positive
4686,content_1d0963b56227,client_4e07408562,20.0,0.00,0.00,keyword article,transactional,1480.0,3445,3,...,280,104,0.09,39.0,20.00,16.67,0,0.956667,7,False Positive
5096,content_f73c382ac270,client_8527a891e2,0.0,0.00,0.00,keyword article,informational,1444.0,1256,4,...,275,104,0.32,15.6,0.00,6.67,1,0.956667,8,Correct
4309,content_41baf0722ad9,client_8527a891e2,0.0,0.00,0.00,keyword article,informational,1596.0,3115,0,...,275,104,0.00,12.8,0.00,25.00,0,0.953333,9,False Positive
2889,content_ee6b25664a03,client_8527a891e2,0.0,0.00,0.00,keyword article,informational,1619.0,691,0,...,223,102,0.00,83.9,0.00,50.00,0,0.953333,10,False Positive


In [38]:
from sklearn.inspection import permutation_importance

# Sample the test set to keep permutation importance reasonably fast
sample_size = min(2000, len(X_test))

sample_indices = X_test.sample(
    n=sample_size,
    random_state=42
).index

X_perm = X_test.loc[sample_indices]
y_perm = y_test.loc[sample_indices]

perm = permutation_importance(
    model,
    X_perm,
    y_perm,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_perm.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

importance_df.head(15)

,feature,importance_mean,importance_std
13,days_with_impressions,0.027809,0.003777
15,content_age_days,0.012522,0.003821
6,clicks_90d,0.004814,0.002866
0,search_volume,0.004196,0.002302
5,impressions_90d,0.002751,0.004895
1,competition,0.002298,0.001811
10,engaged_sessions_90d,0.002281,0.001148
8,sessions_90d,0.001916,0.002153
11,ai_sessions_90d,0.001604,0.000631
18,competition_level,0.001050,0.000706


In [39]:
importance_df.head(10).reset_index(drop=True)

,feature,importance_mean,importance_std
0,days_with_impressions,0.027809,0.003777
1,content_age_days,0.012522,0.003821
2,clicks_90d,0.004814,0.002866
3,search_volume,0.004196,0.002302
4,impressions_90d,0.002751,0.004895
5,competition,0.002298,0.001811
6,engaged_sessions_90d,0.002281,0.001148
7,sessions_90d,0.001916,0.002153
8,ai_sessions_90d,0.001604,0.000631
9,competition_level,0.001050,0.000706


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.